# 02 — Wing Geometry and Target CG

Mean aerodynamic chord, neutral point and the CG location that gives the
chosen static margin. The first cell cross-checks the model against the
rcplanes.online flying-wing CG calculator for the baseline planform
(the same numbers are locked in by `tests/test_wing.py`).

**Datum:** x = 0 at the wing root leading edge, positive aft (mm).


In [ ]:
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "flying_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "flying_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
with open(REPO_ROOT / "mission" / "mission_params.yaml") as f:
    mission = yaml.safe_load(f)
with open(PLATFORM_DIR / "design_point.yaml") as f:
    design = yaml.safe_load(f)
with open(PLATFORM_DIR / "airframe_params.yaml") as f:
    airframe = yaml.safe_load(f)

from uavdesign.atmosphere import G, isa_density, wing_loading_g_dm2, wing_loading_n_m2
from uavdesign.aero import Wing, DragPolar, FixedWingAero

alt_m = mission["mission"]["altitude_asl_m"]
rho = isa_density(alt_m, mission["environment"]["temperature_offset_c"])
wing = Wing.from_dict(airframe["wing"])
aero = design["aero"]
perf = design["performance"]
mass_kg = design["design_point"]["target_auw_kg"]
static_margin = design["design_point"]["static_margin"]
polar = DragPolar.from_oswald(aero["cd0"], wing.aspect_ratio, aero["oswald_e"], aero["cl_max"])

print(f"Site: {alt_m} m ASL, rho = {rho:.4f} kg/m^3")
print(f"Wing: S = {wing.area_dm2:.2f} dm^2, b = {wing.span_mm:.0f} mm, AR = {wing.aspect_ratio:.2f}")
print(f"Polar: CD0 = {polar.cd0}, k = {polar.k:.4f}, CLmax = {polar.cl_max}")


## Planform summary vs calculator


In [ ]:
import pandas as pd

calculator = {   # rcplanes.online, root 275 / tip 107 / half span 350 / sweep 217 / SM 6 %
    "mac_mm": 203.31, "x_le_mac_mm": 92.59, "y_mac_mm": 149.35,
    "x_ac_mm": 143.42, "area_dm2": 13.37, "aspect_ratio": 3.66,
}
summary = wing.summary()
x_cg_target = wing.cg_for_static_margin(static_margin)
summary[f"x_cg_mm @ SM {static_margin:.0%}"] = x_cg_target
calculator[f"x_cg_mm @ SM {static_margin:.0%}"] = 131.22

pd.DataFrame({
    "model": pd.Series(summary).round(2),
    "calculator": pd.Series(calculator),
})


## Planform with MAC, neutral point and target CG

Leading edge at the top, as in the calculator. The shaded band on the
centreline is the usable battery bay.


In [ ]:
y, x = wing.outline()
bay = airframe["battery_bay_x_mm"]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.fill(y, x, color="C0", alpha=0.15)
ax.plot(y, x, color="C0", lw=1.5)
for side in (-1, 1):
    ax.plot([side * wing.y_mac_mm] * 2,
            [wing.x_le_mac_mm, wing.x_le_mac_mm + wing.mac_mm], color="C3", lw=2)
ax.text(wing.y_mac_mm + 6, wing.x_le_mac_mm + wing.mac_mm / 2, "MAC", color="C3", fontsize=9)
ax.axhspan(bay[0], bay[1], xmin=0.47, xmax=0.53, color="C1", alpha=0.35)
ax.axhline(wing.x_np_mm, color="C0", ls="--", lw=1, label=f"neutral point {wing.x_np_mm:.1f} mm")
ax.axhline(x_cg_target, color="k", lw=1.2,
           label=f"target CG {x_cg_target:.1f} mm (SM {static_margin:.0%})")
ax.plot(0, x_cg_target, marker="o", color="k")
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_xlabel("y, spanwise [mm]")
ax.set_ylabel("x, aft of root LE [mm]")
ax.set_title(f"Planform: S = {wing.area_dm2:.2f} dm^2, AR = {wing.aspect_ratio:.2f}, "
             f"LE sweep {wing.placed_panels[0].le_sweep_deg:.1f} deg")
ax.legend(loc="lower center", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_planform_cg.png", dpi=140)
plt.show()


## CG position vs static margin

1 % (expert, light elevator) to 10 % (beginner, stable). Every 1 % of
margin moves the CG forward by 1 % of MAC.


In [ ]:
sms = np.linspace(0.0, 0.12, 25)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(sms * 100, [wing.cg_for_static_margin(s) for s in sms])
ax.axvspan(1, 10, color="C2", alpha=0.08, label="usual range 1-10 %")
ax.axvline(static_margin * 100, color="k", ls="--", lw=1)
ax.set_xlabel("Static margin [% MAC]")
ax.set_ylabel("CG aft of root LE [mm]")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_cg_vs_static_margin.png", dpi=140)
plt.show()


## Scaling the planform to a wing loading

If notebook 01 says the wing is too heavily loaded, a geometric scale of
the whole planform (same shape, same sweep angle) by `sqrt(S_req / S)`
keeps every ratio above and moves the CG proportionally.


In [ ]:
from uavdesign.aero import constraint as con

ws_limit = min(
    con.stall_ws_max(rho, perf["max_stall_speed_m_s"], polar.cl_max),
    con.launch_ws_max(rho, perf["launch_speed_m_s"], polar.cl_max, perf["launch_margin"]),
)
s_req_m2 = mass_kg * G / ws_limit
scale = np.sqrt(max(s_req_m2 / wing.area_m2, 1.0))
print(f"Area needed at {mass_kg:.2f} kg: {s_req_m2*100:.2f} dm^2 (have {wing.area_dm2:.2f})")
print(f"Linear scale factor: {scale:.3f}" + ("  (no change needed)" if scale == 1.0 else ""))


## Outputs handed to notebook 03


In [ ]:
geometry_summary = {
    "mac_mm": wing.mac_mm,
    "x_np_mm": wing.x_np_mm,
    "static_margin": static_margin,
    "x_cg_target_mm": x_cg_target,
}
geometry_summary
